In [32]:
# ============================================
# 1. Import libraries
# ============================================

import pandas as pd
import numpy as np

# Hiển thị đầy đủ cột
pd.set_option("display.max_columns", None)

In [33]:
# ============================================
# 2. Load raw data
# ============================================

train_df = pd.read_csv("../data/raw/raw_data_train.csv")
test_df = pd.read_csv("../data/raw/raw_data_test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (1260428, 19)
Test shape: (315108, 19)


In [34]:
# ============================================
# 3. Chọn các cột cần sử dụng
# ============================================

selected_columns = [
    "price",
    "area",
    "province_name",
    "district_name",
    "ward_name",
    "floor_count",
    "frontage_width",
    "bedroom_count",
    "bathroom_count",
    "house_direction"
]

train_df = train_df[selected_columns]
test_df = test_df[selected_columns]

print("\nSelected columns:")
print(train_df.columns)


Selected columns:
Index(['price', 'area', 'province_name', 'district_name', 'ward_name',
       'floor_count', 'frontage_width', 'bedroom_count', 'bathroom_count',
       'house_direction'],
      dtype='str')


In [35]:
# ============================================
# 4. Kiểm tra missing values
# ============================================

print("\nMissing values in train data:")
print(train_df.isnull().sum())

print("\nMissing values in test data:")
print(test_df.isnull().sum())




Missing values in train data:
price                54996
area                     0
province_name            0
district_name        14489
ward_name           135613
floor_count         740715
frontage_width      491715
bedroom_count       439502
bathroom_count      509368
house_direction    1024949
dtype: int64

Missing values in test data:
price               13726
area                    0
province_name           0
district_name        3599
ward_name           33855
floor_count        185002
frontage_width     122868
bedroom_count      109937
bathroom_count     127075
house_direction    256618
dtype: int64


In [36]:
# ============================================
# 4.1 Tính phần trăm missing values
# ============================================

train_missing_percent = (
    train_df.isnull().sum() / len(train_df)
) * 100

test_missing_percent = (
    test_df.isnull().sum() / len(test_df)
) * 100

# Tạo bảng tổng hợp
train_missing_summary = pd.DataFrame({
    "Missing Count": train_df.isnull().sum(),
    "Missing Percent": train_missing_percent
})

test_missing_summary = pd.DataFrame({
    "Missing Count": test_df.isnull().sum(),
    "Missing Percent": test_missing_percent
})

# Sắp xếp giảm dần theo % missing
train_missing_summary = train_missing_summary.sort_values(
    by="Missing Percent",
    ascending=False
)

test_missing_summary = test_missing_summary.sort_values(
    by="Missing Percent",
    ascending=False
)

print("\nTrain Missing Value Summary:")
print(train_missing_summary)

print("\nTest Missing Value Summary:")
print(test_missing_summary)


Train Missing Value Summary:
                 Missing Count  Missing Percent
house_direction        1024949        81.317537
floor_count             740715        58.766943
bathroom_count          509368        40.412304
frontage_width          491715        39.011748
bedroom_count           439502        34.869267
ward_name               135613        10.759282
price                    54996         4.363280
district_name            14489         1.149530
area                         0         0.000000
province_name                0         0.000000

Test Missing Value Summary:
                 Missing Count  Missing Percent
house_direction         256618        81.438110
floor_count             185002        58.710664
bathroom_count          127075        40.327443
frontage_width          122868        38.992345
bedroom_count           109937        34.888673
ward_name                33855        10.743935
price                    13726         4.355967
district_name             359

In [37]:
# ============================================
# 5. Xóa các dòng bị thiếu giá
# và các dòng có price = 0
# ============================================

print("\nBefore dropping missing and zero price:")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Xóa missing value của price
train_df = train_df.dropna(subset=["price"])
test_df = test_df.dropna(subset=["price"])

# Xóa các dòng có giá bằng 0
train_df = train_df[train_df["price"] > 0]
test_df = test_df[test_df["price"] > 0]

print("\nAfter dropping missing and zero price:")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Kiểm tra lại
print("\nMinimum train price:", train_df["price"].min())
print("Minimum test price:", test_df["price"].min())


Before dropping missing and zero price:
Train shape: (1260428, 10)
Test shape: (315108, 10)

After dropping missing and zero price:
Train shape: (1195187, 10)
Test shape: (298801, 10)

Minimum train price: 1000.0
Minimum test price: 1800000.0


In [38]:
# ============================================
# 6. Xử lý missing values cho numeric columns
# Dùng median vì dữ liệu bất động sản có outlier
# ============================================

numeric_cols = [
    "floor_count",
    "frontage_width",
    "bedroom_count",
    "bathroom_count"
]

for col in numeric_cols:
    median_value = train_df[col].median()

    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)

print("\nMissing values after numeric filling:")
print(train_df[numeric_cols].isnull().sum())


Missing values after numeric filling:
floor_count       0
frontage_width    0
bedroom_count     0
bathroom_count    0
dtype: int64


In [39]:
# ============================================
# 7. Xử lý missing values cho categorical columns
# ============================================

categorical_cols = [
    "district_name",
    "ward_name",
    "house_direction"
]

for col in categorical_cols:
    train_df[col] = train_df[col].fillna("Unknown")
    test_df[col] = test_df[col].fillna("Unknown")

print("\nMissing values after categorical filling:")
print(train_df[categorical_cols].isnull().sum())



Missing values after categorical filling:
district_name      0
ward_name          0
house_direction    0
dtype: int64


In [40]:
# ============================================
# 8. Xóa duplicate rows
# ============================================

print("\nBefore removing duplicates:")
print("Train:", train_df.shape)
print("Test:", test_df.shape)

train_df = train_df.drop_duplicates()
test_df = test_df.drop_duplicates()

print("\nAfter removing duplicates:")
print("Train:", train_df.shape)
print("Test:", test_df.shape)


Before removing duplicates:
Train: (1195187, 10)
Test: (298801, 10)

After removing duplicates:
Train: (976848, 10)
Test: (279226, 10)


In [41]:
# ============================================
# 9. Xử lý outliers cho price
# Giữ dữ liệu từ 1% đến 99%
# ============================================

q1 = train_df["price"].quantile(0.01)
q99 = train_df["price"].quantile(0.99)

train_df = train_df[
    (train_df["price"] >= q1) &
    (train_df["price"] <= q99)
]

print("\nAfter removing outliers:")
print("Train shape:", train_df.shape)


After removing outliers:
Train shape: (957410, 10)


In [42]:
# ============================================
# 10. Xử lý outliers cho area
# ============================================

q1_area = train_df["area"].quantile(0.01)
q99_area = train_df["area"].quantile(0.99)

train_df = train_df[
    (train_df["area"] >= q1_area) &
    (train_df["area"] <= q99_area)
]

print("\nAfter removing area outliers:")
print("Train shape:", train_df.shape)




After removing area outliers:
Train shape: (940265, 10)


In [43]:
# ============================================
# 11. Log transform target
# Giúp dữ liệu phân phối tốt hơn
# ============================================

train_df["price"] = np.log1p(train_df["price"])
test_df["price"] = np.log1p(test_df["price"])

print("\nPrice transformed using log1p")


Price transformed using log1p


In [44]:
# ============================================
# 12. Reset index
# ============================================

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [45]:
# ============================================
# 13. Kiểm tra dữ liệu sau cleaning
# ============================================

print("\nTrain info:")
print(train_df.info())

print("\nTest info:")
print(test_df.info())

print("\nTrain describe:")
print(train_df.describe())


Train info:
<class 'pandas.DataFrame'>
RangeIndex: 940265 entries, 0 to 940264
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   price            940265 non-null  float64
 1   area             940265 non-null  float64
 2   province_name    940265 non-null  str    
 3   district_name    940265 non-null  str    
 4   ward_name        940265 non-null  str    
 5   floor_count      940265 non-null  float64
 6   frontage_width   940265 non-null  float64
 7   bedroom_count    940265 non-null  float64
 8   bathroom_count   940265 non-null  float64
 9   house_direction  940265 non-null  str    
dtypes: float64(6), str(4)
memory usage: 106.0 MB
None

Test info:
<class 'pandas.DataFrame'>
RangeIndex: 279226 entries, 0 to 279225
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   price            279226 non-null  float64
 1   area      

In [46]:
# ============================================
# 14. Kiểm tra missing values cuối cùng
# ============================================

print("\nFinal missing values in train:")
print(train_df.isnull().sum())

print("\nFinal missing values in test:")
print(test_df.isnull().sum())


Final missing values in train:
price              0
area               0
province_name      0
district_name      0
ward_name          0
floor_count        0
frontage_width     0
bedroom_count      0
bathroom_count     0
house_direction    0
dtype: int64

Final missing values in test:
price              0
area               0
province_name      0
district_name      0
ward_name          0
floor_count        0
frontage_width     0
bedroom_count      0
bathroom_count     0
house_direction    0
dtype: int64


In [16]:
# ============================================
# 15. Save clean data
# ============================================

train_df.to_csv(
    "../data/clean/clean_data_train.csv",
    index=False
)

test_df.to_csv(
    "../data/clean/clean_data_test.csv",
    index=False
)

print("\nCleaned data saved successfully!")


Cleaned data saved successfully!


In [17]:
# ============================================
# 16. Final shapes
# ============================================

print("\nFinal dataset shapes:")
print("Train:", train_df.shape)
print("Test:", test_df.shape)


Final dataset shapes:
Train: (940265, 10)
Test: (279226, 10)
